In [2]:
%pip install llama-index-vector-stores-lancedb
%pip install llama-index-multi-modal-llms-openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# %pip install llama-index-multi-modal-llms-openai
# %pip install llama-index-vector-stores-lancedb
# %pip install llama-index-embeddings-clip

In [4]:
# %pip install llama_index ftfy regex tqdm
# %pip install -U openai-whisper
# %pip install git+https://github.com/openai/CLIP.git
# %pip install torch torchvision
# %pip install matplotlib scikit-image
# %pip install lancedb
# %pip install moviepy
# # %pip install pytube
# %pip install yt_dlp
# %pip install pydub
# %pip install SpeechRecognition
# %pip install ffmpeg-python
# %pip install soundfile

In [5]:
from moviepy.video.io.VideoFileClip import VideoFileClip
from pathlib import Path
import speech_recognition as sr
# from pytube import YouTube
from pprint import pprint

In [6]:
from dotenv import load_dotenv
import os
load_dotenv()
pinecone_api_key = os.environ.get('PINECONE_API_KEY')

In [7]:
video_url = "https://www.youtube.com/watch?v=d_qvLDhkg00"
output_video_path = "./video_data/"
output_folder = "./mixed_data/"
output_audio_path = "./mixed_data/output_audio.wav"

filepath = output_video_path + "input_vid.mp4"
Path(output_folder).mkdir(parents=True, exist_ok=True)

In [8]:
# !pip install yt_dlp

In [9]:
from PIL import Image
import matplotlib.pyplot as plt
import os

#추후 image retrieval 결과 플라팅 용
def plot_images(image_paths):
    images_shown = 0
    plt.figure(figsize=(16, 9))
    for img_path in image_paths:
        if os.path.isfile(img_path):
            image = Image.open(img_path)

            plt.subplot(2, 3, images_shown + 1)
            plt.imshow(image)
            plt.xticks([])
            plt.yticks([])

            images_shown += 1
            if images_shown >= 7:
                break

In [10]:
import yt_dlp
from pathlib import Path

def download_video(url, output_path):
    Path(output_video_path).mkdir(parents=True, exist_ok=True)

    ydl_opts = {
        "outtmpl": f"{output_video_path}/input_vid.%(ext)s",
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)
        metadata = {
            "Author": info.get("uploader"),
            "Title": info.get("title"),
            "Views": info.get("view_count"),
        }
    return metadata


/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [11]:
# 400 Bad Request 발생
# def download_video(url, output_path):
#     yt = YouTube(url)
#     metadata = {"Author": yt.author, "Title": yt.title, "Views": yt.views}
#     yt.streams.get_highest_resolution().download(
#         output_path=output_path, filename="input_vid.mp4"
#     )
#     return metadata


def video_to_images(video_path, output_folder):
    clip = VideoFileClip(video_path)
    clip.write_images_sequence(
        os.path.join(output_folder, "frame%04d.png"), fps=0.2
    )


def video_to_audio(video_path, output_audio_path):
    clip = VideoFileClip(video_path)
    audio = clip.audio
    audio.write_audiofile(output_audio_path)


def audio_to_text(audio_path):
    recognizer = sr.Recognizer()
    audio = sr.AudioFile(audio_path)

    with audio as source:
        audio_data = recognizer.record(source)

        try:
            text = recognizer.recognize_whisper(audio_data)
        except sr.UnknownValueError:
            print("Speech recognition could not understand the audio.")
        except sr.RequestError as e:
            print(f"Could not request results from service; {e}")

    return text

In [ ]:
try:
    # 영상 다운 전
    metadata_vid = download_video(video_url, output_video_path)

    # 영상 다운 후 
    # filepath = str(Path(output_video_path) / "input_vid.mp4")
    # assert Path(filepath).exists(), f"파일 없음: {filepath}"
    # metadata_vid = {'Author': '3Blue1Brown', 'Title': 'A pretty reason why Gaussian + Gaussian = Gaussian', 'Views': 926302}

    video_to_images(filepath, output_folder)
    video_to_audio(filepath, output_audio_path)
    text_data = audio_to_text(output_audio_path)

    with open(output_folder + "output_text.txt", "w") as file:
        file.write(text_data)
    print("Text data saved to file")

    file.close()
    os.remove(output_audio_path)
    print("Audio file removed")

except Exception as e:
    raise e

In [19]:

from llama_index.core import StorageContext
from llama_index.vector_stores.lancedb import LanceDBVectorStore

text_store = LanceDBVectorStore(uri="lancedb", table_name="text_collection")
image_store = LanceDBVectorStore(uri="lancedb", table_name="image_collection")
storage_context = StorageContext.from_defaults(
    vector_store=text_store, image_store=image_store
)

from llama_index.core import SimpleDirectoryReader

# -> OpenAI CLIP 모델로 Text, Image Embedding 가해짐
documents = SimpleDirectoryReader(output_folder).load_data()

from llama_index.core.indices import MultiModalVectorStoreIndex

index = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

2026-03-05 15:32:45,177 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [ ]:
# metadata_vid = {'Author': '3Blue1Brown', 'Title': 'A pretty reason why Gaussian + Gaussian = Gaussian', 'Views': 926302}

In [ ]:

# from llama_index.core.response.notebook_utils import display_source_node
# from llama_index.core.schema import ImageNode

# def retrieve(retriever_engine, query_str):
#     retrieval_results = retriever_engine.retrieve(query_str)

#     retrieved_image = []
#     retrieved_text = []
#     for res_node in retrieval_results:
#         if isinstance(res_node.node, ImageNode):
#             retrieved_image.append(res_node.node.metadata["file_path"])
#         else:
#             display_source_node(res_node, source_length=200)
#             retrieved_text.append(res_node.text)

#     return retrieved_image, retrieved_text

In [ ]:
from llama_index.core.response.notebook_utils import display_source_node
from llama_index.core.schema import ImageNode

def retrieve(index, query_str, text_top_k=5, image_top_k=5, show_source=True):
    r = index.as_retriever(
        similarity_top_k=text_top_k,
        image_similarity_top_k=image_top_k,
    )

    text_nodes = r.text_retrieve(query_str)
    image_nodes = r.text_to_image_retrieve(query_str)

    retrieved_text = []
    for n in text_nodes:
        if show_source:
            display_source_node(n, source_length=200)
        retrieved_text.append(n.text)

    retrieved_image = []
    for n in image_nodes:
        if isinstance(n.node, ImageNode):
            fp = (n.node.metadata or {}).get("file_path")
            if fp:
                retrieved_image.append(fp)

    print(f"text={len(retrieved_text)}, image={len(retrieved_image)}")
    return retrieved_image, retrieved_text


In [ ]:
query_str = "Using examples from video, explain all things covered in the video regarding the gaussian function"

img, txt= retrieve(index, query_str)

In [78]:
img

['/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0033.png',
 '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0033.png',
 '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0033.png',
 '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0032.png',
 '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0032.png']

In [ ]:
context_str = "".join(txt)
context_str

In [ ]:
plot_images(img)

In [70]:
image_documents = SimpleDirectoryReader(
    input_dir=output_folder, input_files=img
).load_data()

In [71]:
import base64
from llama_index.core.schema import ImageDocument, ImageNode
from llama_index.core.base.llms.types import ImageBlock

def to_supported_image(doc):
    if isinstance(doc, (ImageNode, ImageBlock)):
        return doc

    if isinstance(doc, ImageDocument):
        # ImageDocument -> ImageNode
        return ImageNode(
            text=doc.text or "",
            metadata=doc.metadata,
            image=doc.image,                    # base64 string or None
            image_path=doc.image_path,
            image_url=doc.image_url,
            image_mimetype=doc.image_mimetype,
        )

    raise TypeError(f"Unsupported image doc type: {type(doc)}")

converted_images = [to_supported_image(d) for d in image_documents]

In [76]:
converted_images

[ImageNode(id_='91834bb8-571c-472b-8665-60430073071c', embedding=None, metadata={'file_path': '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0033.png', 'file_name': 'frame0033.png', 'file_type': 'image/png', 'file_size': 462037, 'creation_date': '2026-03-05', 'last_modified_date': '2026-03-05'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', image=None, image_path='/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/frame0033.png', image_url=None, image_mimetype='image/png', text_embedding=None),
 ImageNode(id_='dba6c028-d665-4359-9f99-f0614ca444fe', embedding=None, metadata={'file_path': '/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/05.VectorDB_Advanced/Multimodal/mixed_data/fram

In [77]:

import json

metadata_str = json.dumps(metadata_vid)

qa_tmpl_str = (
    "Given the provided information, including relevant images and retrieved context from the video, \
 accurately and precisely answer the query without any additional prior knowledge.\n"
    "Please ensure honesty and responsibility, refraining from any racist or sexist remarks.\n"
    "---------------------\n"
    "Context: {context_str}\n"
    "Metadata for video: {metadata_str} \n"
    "---------------------\n"
    "Query: {query_str}\n"
    "Answer: "
)

In [81]:

from llama_index.multi_modal_llms.openai import OpenAIMultiModal

openai_mm_llm = OpenAIMultiModal(
    model="gpt-4o-mini",   # gpt-4-vision-preview 대신 권장
    max_new_tokens=1500,
)

response_1 = openai_mm_llm.complete(
    prompt=qa_tmpl_str.format(
        context_str=context_str, query_str=query_str, metadata_str=metadata_str
    ),
    image_documents=converted_images,
)


/var/folders/95/t8symb5521g2916cz4qtrsjc0000gn/T/ipykernel_64913/3989695562.py:3: DeprecationWarning: Call to deprecated class OpenAIMultiModal. (The package has been deprecated and will no longer be maintained. Please use llama-index-llms-openai (preferably the Responses API) instead. See Multi Modal LLMs documentation for a complete guide on migration: https://docs.llamaindex.ai/en/stable/understanding/using_llms/using_llms/#multi-modal-llms) -- Deprecated since version 0.5.2.
  openai_mm_llm = OpenAIMultiModal(
2026-03-05 16:16:39,342 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [82]:
pprint(response_1.text)

('The video covers several key concepts related to the Gaussian function and '
 'its significance in probability theory, particularly in relation to the '
 'central limit theorem. Here’s a summary of the main points discussed:\n'
 '\n'
 '1. **Basic Function of Gaussian**: The Gaussian function is represented as '
 '\\( e^{-x^2} \\). This function is symmetric and smooth, with its mass '
 'concentrated towards the center, making it a natural choice for modeling '
 'distributions in probability.\n'
 '\n'
 '2. **Central Limit Theorem (CLT)**: The video emphasizes the central limit '
 'theorem, which states that as you sum multiple independent random variables, '
 'their distribution approaches a normal distribution (Gaussian) under certain '
 'conditions. This is illustrated with examples like rolling a weighted die '
 'multiple times.\n'
 '\n'
 '3. **Convolution of Gaussian Functions**: The main focus is on computing the '
 'convolution of two Gaussian functions. The convolution operatio